In [ ]:
# End‑to‑End PEFT example with LoRA and BERT (tweet sentiment)
# Install (VS Code terminal)
#   %pip install -U transformers datasets peft accelerate torch

from typing import Dict
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
import evaluate

# ── Data ───────────────────────────────────────────────────────────────────────
dataset = load_dataset("tweet_eval", "sentiment")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tok(batch: Dict[str, str]):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

encoded = dataset.map(tok, batched=True)  # type: ignore[arg-type]
encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
train_ds, val_ds, test_ds = (
    encoded["train"],
    encoded["validation"],
    encoded["test"],
)

# ── Model + PEFT (LoRA) ─────────────────────────────────────────────────────────
base_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(base_model, lora_cfg)  # freezes backbone, adds adapters

# ── Trainer ─────────────────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir="results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="accuracy",
    load_best_model_at_end=True,
    logging_steps=100,
)

accuracy = evaluate.load("accuracy")

def metrics(pred):
    logits, labels = pred
    return accuracy.compute(predictions=logits.argmax(-1), references=labels)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=metrics,
)

trainer.train()
print(trainer.evaluate(test_ds))

# ── Save tiny LoRA adapter + tokenizer ──────────────────────────────────────────
model.save_pretrained("results/lora-adapter")
tokenizer.save_pretrained("results/tokenizer")

# ── Inference demo ──────────────────────────────────────────────────────────────
text = "I love how easy parameter‑efficient fine‑tuning makes my life!"
inputs = tokenizer(text, return_tensors="pt").to(model.device)
label_map = {0: "negative", 1: "neutral", 2: "positive"}
print("Prediction →", label_map[model(**inputs).logits.argmax(-1).item()])


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (c:\Users\Jamie\AppData\Local\Programs\Python\Python312\Lib\site-packages\accelerate\utils\memory.py)